# This notebook is for unit tests on tools and models

In [1]:
import pandas as pd
import numpy as np
import time
from config import TRADE_LIST
from src.utils.CalFactorFramework import FactorCalculator, FactorRegistry

## test CalFactorFramework.py

In [2]:
from src.utils.factors.rv_daily import compute as rv_daily

In [3]:
calc = FactorCalculator()
registry = FactorRegistry(calc, "./register/factor_registry.json")

2025-10-10 21:46:28 [INFO] FactorRegistry: Loaded metadata for 1 factors from register\factor_registry.json


Loaded metadata for 1 factors from register\factor_registry.json


In [4]:
register_config = {'name': 'rv_daily', 'factor_func': rv_daily, 'frequency': 'min',
                   'fields': ["symbol", "timestamp", "Close"],
                   'type_': 'alpha', 'category': 'volatility', 'description': 'test'}

registry.register(**register_config) # 使用默认数据注册
# registry.register(**register_config, test=1) register_config之后添加的是动态参数

2025-10-10 21:46:30 [INFO] FactorRegistry: Factor 'rv_daily' registered: src.utils.factors.rv_daily.compute


In [5]:
registry.list_factors()

{'rv_daily': {'description': 'test',
  'frequency': 'min',
  'fields': ['symbol', 'timestamp', 'Close'],
  'created_time': '2025-10-10T21:46:30.950572',
  'updated_time': '2025-10-10T21:46:30.950572',
  'type': 'alpha',
  'category': 'volatility',
  'has_function': True}}

In [37]:
test11 = registry._factors['rv_daily']
print(test11.get('func_qualname') is None)

False


log有输出，说明已经被注册

In [8]:
# test direct calculation
start_single_process = time.time()
registry.calculate(
        'rv_daily',
        is_batch=True,
        is_parallel=False,
        symbols=TRADE_LIST,
        price_col='Close'  # 这里的是动态参数，根据因子计算函数要求给
    )
end_single_process = time.time()
print(f"single-process takes {end_single_process - start_single_process} seconds")
res = registry.get_factor_data('rv_daily')
print(res.head())
print("===============================")
print(res.shape)

Processing Date Batch 1/2: 20250101 to 20250629
Processing Date Batch 2/2: 20250630 to 20250910
single-process takes 255.25425601005554 seconds


FactorRegistryError: Calculator does not support 'get_factor_data' method

In [9]:
# test multi-process
start_multi_process = time.time()
registry.calculate(
        'rv_daily',
        is_batch=True,
        is_parallel=True,
        symbols=TRADE_LIST,
        batch_size=120, # 每个增量的容量
        parallel_batch_size=20, # 每个进程要处理多少天数据
        n_jobs=6
    )  # 增量是处于内存考虑，记一次同时读取多少填数据，如果内存够大，batch_size直接取天数也可以尝试
end_multi_process = time.time()
print(f"multi-process takes {end_multi_process - start_multi_process} seconds")
res1 = registry.get_factor_data('rv_daily')
print(res1.head())
print("===============================")
print(res1.shape)

Processing Date Batch 1/7: 20250101 to 20250209
Processing Date Batch 2/7: 20250210 to 20250321
Processing Date Batch 3/7: 20250322 to 20250430
Processing Date Batch 4/7: 20250501 to 20250609
Processing Date Batch 5/7: 20250610 to 20250719
Processing Date Batch 6/7: 20250720 to 20250828
Processing Date Batch 7/7: 20250829 to 20250910
multi-process takes 255.25425601005554 seconds


FactorRegistryError: Calculator does not support 'get_factor_data' method

## test BacktestTool.py

In [ ]:
from src.utils import BacktestTools


In [4]:
test = pd.read_parquet('./data/factors/alpha/factor_rev_short.parquet')

In [12]:
for group in np.log(test['rev_short']).groupby(test["symbol"], sort=False):
    print(group)

D:\DevTools\CondaBases\py310_base\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
D:\DevTools\CondaBases\py310_base\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


('1000000MOGUSDT', 0          -inf
1          -inf
2          -inf
3          -inf
4          -inf
           ... 
33931925    NaN
33931926    NaN
33931927    NaN
33931928    NaN
33931929    NaN
Name: rev_short, Length: 105088, dtype: float64)
('1000BONKUSDT', 1642            -inf
1643            -inf
1644            -inf
1645            -inf
1646            -inf
              ...   
33933567         NaN
33933568         NaN
33933569   -8.161359
33933570   -6.509304
33933571   -6.979085
Name: rev_short, Length: 105088, dtype: float64)
('1000CATUSDT', 3284            -inf
3285            -inf
3286            -inf
3287            -inf
3288            -inf
              ...   
33935209         NaN
33935210   -7.380658
33935211   -8.340825
33935212   -7.193884
33935213   -6.408306
Name: rev_short, Length: 105088, dtype: float64)
('1000CHEEMSUSDT', 4926       -inf
4927       -inf
4928       -inf
4929       -inf
4930       -inf
           ... 
33936851    NaN
33936852    NaN
33936853    NaN


In [13]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("FactorRegistry")

In [14]:
logger

<Logger FactorRegistry (INFO)>

In [8]:
pd.read_parquet('./data/hour_data/all_data.parquet')

,index,symbol,timestamp,Open,High,Low,Close,Volume,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume
0,0,1000000MOGUSDT,2025-01-01 00:00:00,2.0176,2.0499,2.0144,2.0494,190819.3,387521.97478,4242,96328.9,195595.79399
1,1,1000000MOGUSDT,2025-01-01 01:00:00,2.0495,2.0539,2.0322,2.0340,93330.7,190936.77098,2618,35284.2,72176.06363
2,2,1000000MOGUSDT,2025-01-01 02:00:00,2.0339,2.0432,2.0266,2.0417,98181.5,199724.19965,3291,48935.1,99556.09385
3,3,1000000MOGUSDT,2025-01-01 03:00:00,2.0419,2.0433,2.0205,2.0225,77931.6,158096.89242,2398,34886.3,70782.02433
4,4,1000000MOGUSDT,2025-01-01 04:00:00,2.0218,2.0230,2.0019,2.0024,256532.6,516197.94787,4636,98522.2,198343.83696
...,...,...,...,...,...,...,...,...,...,...,...,...
2204131,113251,ZRXUSDT,2025-09-10 19:00:00,0.2782,0.2784,0.2732,0.2744,1185208.1,326047.35412,3397,501308.4,137856.89024
2204132,113252,ZRXUSDT,2025-09-10 20:00:00,0.2744,0.2764,0.2741,0.2760,584202.2,160860.18278,2057,301499.3,83021.44949
2204133,113253,ZRXUSDT,2025-09-10 21:00:00,0.2760,0.2785,0.2758,0.2782,652458.9,181072.66377,1936,312065.0,86560.13083
2204134,113254,ZRXUSDT,2025-09-10 22:00:00,0.2782,0.2799,0.2779,0.2794,979806.5,273252.33191,2476,416981.4,116294.91660


1. factors(ew): factor/dl factor 10 --> multi-factors model
                                        --> 4 hours/6 hour/... return
                                        long only (short 30%)? long short (short BTC/ETH/... market index)?

2. optimizer(position/factor weighting): risk model-->Barra, control exposure; dl/xgboost/black litterman model
3. multi-factor + optimizer